# 09. Evaluasi final dan output

AI-013. Hasil satu final setelah kandidat dikunci. Sel membaca bukti aktual, tanpa training atau final rerun. Notebook08 mempertahankan snapshot development sebelumnya.

In [1]:
from pathlib import Path
import sys,json,hashlib
import pandas as pd
from IPython.display import display
ROOT=Path.cwd()
if ROOT.name=='notebooks':ROOT=ROOT.parent
read=lambda path:json.loads((ROOT/path).read_text(encoding='utf-8'))
print('Project interpreter:',sys.executable)

Project interpreter: F:\Generative_AI\Vibe_Coding_Proyek_1\.venv\Scripts\python.exe


## 1. Candidate selection before final

Best ordinary CE after completed fixed budget, then development gate. Never choose a new checkpoint from final failures.

In [2]:
selection=read('reports/training/TRAIN-015_FINAL_SELECTION.json')
display(selection)

{'recorded_at_utc': '2026-10-13T12:18:07.153281+00:00',
 'run_id': 'TRAIN-015',
 'checkpoint': 'checkpoints/TRAIN-015/best.pt',
 'checkpoint_sha256': '3399709ffe2fa7981c9d26b8ff5723f7821bd78e302bea6dfe63aaf1d27c8638',
 'selected_by': 'best full validation ordinary CE after fixed budget; development generation gate',
 'training_report_sha256': '8e9e0784d3139cdfd82e42cc9918f328820af2a1ce68c889f683ff8682be98fb',
 'development_report_sha256': 'b253c09eda4bad2532ae8583d0840064434fbb62e2f1059adcca1dc38f6064ca',
 'policy_sha256': 'eafd347cc960f4eefdaee5c37b1e276328b77b97b246944a23c2c97b3506eb2c',
 'final_suite': {'path': 'data\\processed\\curriculum\\v6\\final_prompts.jsonl',
  'sha256': '3535c7c954fedefcf46911c059fc94171fdb689d2bd8982cbf4e20580c3858b5',
  'count': 48},
 'acceptance': {'schema_valid_rate': 0.95,
  'exact_contract_rate': 0.9,
  'repair_allowed': False},
 'decode': {'precision': 'bf16',
  'temperature': 0,
  'seed': 42,
  'max_new_tokens': 768},
 'final_targets_read_for_selecti

## 2. Raw final measurement and hashes

Final now consumed. No JSON repair or target substitution.

In [3]:
final=read('reports/training/TRAIN-015-FINAL_curriculum_evaluation.json')
gate=read('reports/training/TRAIN-015_FINAL_GATE.json')
assert final['checkpoint_sha256']==selection['checkpoint_sha256']
assert final['prompt_file_sha256']==selection['final_suite']['sha256']
assert hashlib.sha256((ROOT/final['outputs_path']).read_bytes()).hexdigest()==final['outputs_sha256']
display({k:final[k] for k in ('schema_valid','exact_contract','count','resolved_links','all_ended_eos','contract_gate_passed','token_metrics')})

{'schema_valid': 48,
 'exact_contract': 44,
 'count': 48,
 'resolved_links': 48,
 'all_ended_eos': True,
 'contract_gate_passed': True,
 'token_metrics': {'loss': 0.0029893913076406266,
  'perplexity': 1.0029938639935947,
  'valid_tokens': 624225,
  'batches': 360}}

## 3. Every failed output remains visible

Comparisons are for reporting after final. They are not additional train labels in this session.

In [4]:
display(gate['failures'])
raw=read(final['outputs_path'])
display(pd.DataFrame([{'recipe':r['recipe'],'style':r['target_spec']['style_mode'],
    'expected_name':r['target_spec']['title'],'actual_name':(r['parsed_spec'] or {}).get('title'),
    'exact':r['exact_contract'],'EOS':r['stop_reason']=='eos'} for r in raw]))

[{'example_id': 'a441662cd6f76613b5445b14b14481db81a5b3603f8e9257421857e70e5f99bc',
  'recipe': 'article',
  'style_mode': 'plain_css',
  'differences': [{'path': '$.sections[0].props.brand',
    'expected': 'Kursus Smartgismartni',
    'actual': 'Kursus Smartewelmartni'},
   {'path': '$.sections[2].props.text',
    'expected': 'Kursus Smartgismartni',
    'actual': 'Kursus Smartewelmartni'},
   {'path': '$.title',
    'expected': 'Kursus Smartgismartni',
    'actual': 'Kursus Smartewelmartni'}]},
 {'example_id': '1a8df8a21bcb5a00092c6a17fa123820577f672eab06f5dd8736b258b85eb922',
  'recipe': 'pricing',
  'style_mode': 'tailwind',
  'differences': [{'path': '$.sections[0].props.brand',
    'expected': 'Kursus Smartgismartni',
    'actual': 'Kursus Smartdiremartni'},
   {'path': '$.sections[1].props.heading',
    'expected': 'Kursus Smartgismartni',
    'actual': 'Kursus Smartdiremartni'},
   {'path': '$.sections[3].props.text',
    'expected': 'Kursus Smartgismartni',
    'actual': 'Kur

,recipe,style,expected_name,actual_name,exact,EOS
0,profile,plain_css,Kafe Webtechraw,Kafe Webtechraw,True,True
1,profile,plain_css,Pustaka Nostargomi,Pustaka Nostargomi,True,True
2,profile,tailwind,Kafe Webtechraw,Kafe Webtechraw,True,True
3,profile,tailwind,Klinik Sunpiblue,Klinik Sunpiblue,True,True
4,features,plain_css,Studio Bagreenlo,Studio Bagreenlo,True,True
5,features,plain_css,Dapur Fakamaweb,Dapur Fakamaweb,True,True
6,features,tailwind,Klinik Takacraft,Klinik Takacraft,True,True
7,features,tailwind,Dapur Fakamaweb,Dapur Fakamaweb,True,True
8,article,plain_css,Kursus Smartgismartni,Kursus Smartewelmartni,False,True
9,article,plain_css,Pustaka Nostargomi,Pustaka Nostargomi,True,True


## 4. Renderer artifacts

Only strict accepted raw outputs are exported. Local CSS/JavaScript packaging is not browser proof.

In [5]:
if gate['gallery_report']:
    gallery=read(gate['gallery_report'])
    display({k:gallery[k] for k in ('output_count','output_dir','archive_sha256','rejected','browser_evaluated')})
    display(pd.DataFrame([{'title':e['title'],'recipe':e['recipe'],'style':e['style_mode'],'directory':e['directory']} for e in gallery['entries']]))
else:
    print('No passed-final gallery exported.')

{'output_count': 44,
 'output_dir': 'examples\\generated\\AI-013',
 'archive_sha256': 'd3d163bed1d7eed193320f24e0cc0ce9e1cb1675afc400f24afcfc61f7fbcf76',
 'rejected': [{'example_id': 'a441662cd6f76613b5445b14b14481db81a5b3603f8e9257421857e70e5f99bc',
   'contract': {'checked': True,
    'requested': {'title': 'Kursus Smartgismartni',
     'style_mode': 'plain_css',
     'primary': '#2563eb',
     'font': 'mono',
     'text': '#111827',
     'background': '#ffffff',
     'main_text': 'Kami mendukung tim kreatif untuk mengelola layanan secara mudah.'},
    'errors': ["title: diminta 'Kursus Smartgismartni', model menghasilkan 'Kursus Smartewelmartni'",
     'Section nav: isi tidak sesuai kontrak prompt',
     'Section footer: isi tidak sesuai kontrak prompt']}},
  {'example_id': '1a8df8a21bcb5a00092c6a17fa123820577f672eab06f5dd8736b258b85eb922',
   'contract': {'checked': True,
    'requested': {'title': 'Kursus Smartgismartni',
     'style_mode': 'tailwind',
     'primary': '#10b981',
 

,title,recipe,style,directory
0,Kafe Webtechraw,profile,plain_css,examples\generated\AI-013\01-profile-plain_css
1,Pustaka Nostargomi,profile,plain_css,examples\generated\AI-013\02-profile-plain_css
2,Kafe Webtechraw,profile,tailwind,examples\generated\AI-013\03-profile-tailwind
3,Klinik Sunpiblue,profile,tailwind,examples\generated\AI-013\04-profile-tailwind
4,Studio Bagreenlo,features,plain_css,examples\generated\AI-013\05-features-plain_css
5,Dapur Fakamaweb,features,plain_css,examples\generated\AI-013\06-features-plain_css
6,Klinik Takacraft,features,tailwind,examples\generated\AI-013\07-features-tailwind
7,Dapur Fakamaweb,features,tailwind,examples\generated\AI-013\08-features-tailwind
8,Pustaka Nostargomi,article,plain_css,examples\generated\AI-013\10-article-plain_css
9,Studio Bagreenlo,article,tailwind,examples\generated\AI-013\11-article-tailwind


## 5. Next phase and honest readiness

Controlled contracts are distinct from general prompts, reviewed human SFT and browser acceptance.

In [6]:
display({k:gate[k] for k in ('status','selected_runtime','exported_websites','browser_evaluated','generator_ui_implemented','limits')})

{'status': 'controlled-contract-passed',
 'selected_runtime': 'TRAIN-015/best',
 'exported_websites': 44,
 'browser_evaluated': False,
 'generator_ui_implemented': False,
 'limits': ['Synthetic shared recipes with heldout fictional brands; arbitrary prompts/layouts not established',
  'This final is now consumed; future tuning requires a new independent final',
  'No retraining or checkpoint reselection from this result in AI-013',
  'Browser review and application UI remain next stages']}